# vtp-eval — POPE benchmark on LLaVA-1.5 7B
Compares baseline + 5 visual-token-pruning methods (FastV, SparseVLMs, VisionZip, DivPrune, SparseVILA).
**Runtime:** Colab Pro, GPU L4 (24 GB). **Total time:** ~3–4 hours for full POPE.
**IMPORTANT:** You MUST manually restart the runtime between methods (marked in red below).


In [ ]:
# Cell 2 — Clone vtp-eval workspace
!git clone --depth 1 https://github.com/<GITHUB_USER>/vtp-eval.git /content/vtp-eval || (cd /content/vtp-eval && git pull)
%cd /content/vtp-eval


In [ ]:
# Cell 3 — Detect GPU + set L4 config
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
os.environ["VTP_BATCH_SIZE"] = "2"
os.environ["VTP_DTYPE"] = "float16"
print("Batch size:", os.environ["VTP_BATCH_SIZE"])


In [ ]:
# Cell 4 — HuggingFace login (need for POPE dataset + LLaVA weights)
from huggingface_hub import notebook_login
notebook_login()


## Run 1/6 — baseline
No restart needed for the first run.

In [ ]:
# Cell B — Install + run baseline
%cd /content/vtp-eval
!bash install/baseline.sh
!bash scripts/run_one.sh baseline


## Run 2/6 — FastV (K=2, R=128)
**🔴 BEFORE running these cells: manually restart the runtime** (Runtime → Restart runtime), then run **Cell A** to force-restart, **Cell B** to install + run.

This is required because previous methods may have patched `transformers.models.llama.modeling_llama` in ways that conflict with this method.


In [ ]:
# Cell A — Force kernel restart (alternative to manual restart)
import os; os._exit(0)


In [ ]:
# Cell B — Install + run FastV (K=2, R=128)
%cd /content/vtp-eval
!bash install/fastv.sh
!bash scripts/run_one.sh fastv_K2_R128


## Run 3/6 — SparseVLMs (retain=192)
**🔴 BEFORE running these cells: manually restart the runtime** (Runtime → Restart runtime), then run **Cell A** to force-restart, **Cell B** to install + run.

This is required because previous methods may have patched `transformers.models.llama.modeling_llama` in ways that conflict with this method.


In [ ]:
# Cell A — Force kernel restart (alternative to manual restart)
import os; os._exit(0)


In [ ]:
# Cell B — Install + run SparseVLMs (retain=192)
%cd /content/vtp-eval
!bash install/sparsevlm.sh
!bash scripts/run_one.sh sparsevlm_192


## Run 4/6 — VisionZip (64 tokens)
**🔴 BEFORE running these cells: manually restart the runtime** (Runtime → Restart runtime), then run **Cell A** to force-restart, **Cell B** to install + run.

This is required because previous methods may have patched `transformers.models.llama.modeling_llama` in ways that conflict with this method.


In [ ]:
# Cell A — Force kernel restart (alternative to manual restart)
import os; os._exit(0)


In [ ]:
# Cell B — Install + run VisionZip (64 tokens)
%cd /content/vtp-eval
!bash install/visionzip.sh
!bash scripts/run_one.sh visionzip_64


## Run 5/6 — DivPrune (ratio=0.098)
**🔴 BEFORE running these cells: manually restart the runtime** (Runtime → Restart runtime), then run **Cell A** to force-restart, **Cell B** to install + run.

This is required because previous methods may have patched `transformers.models.llama.modeling_llama` in ways that conflict with this method.


In [ ]:
# Cell A — Force kernel restart (alternative to manual restart)
import os; os._exit(0)


In [ ]:
# Cell B — Install + run DivPrune (ratio=0.098)
%cd /content/vtp-eval
!bash install/divprune.sh
!bash scripts/run_one.sh divprune_0.098


## Run 6/6 — SparseVILA (enc=0.5, dec=0.5)
**🔴 BEFORE running these cells: manually restart the runtime** (Runtime → Restart runtime), then run **Cell A** to force-restart, **Cell B** to install + run.

This is required because previous methods may have patched `transformers.models.llama.modeling_llama` in ways that conflict with this method.


In [ ]:
# Cell A — Force kernel restart (alternative to manual restart)
import os; os._exit(0)


In [ ]:
# Cell B — Install + run SparseVILA (enc=0.5, dec=0.5)
%cd /content/vtp-eval
!bash install/sparsevila.sh
!bash scripts/run_one.sh sparsevila_0.5_0.5


## Aggregate & visualize
After all 6 runs complete, this cell builds `summary.csv` and plots the two
canonical scatter plots (accuracy vs FLOPs, accuracy vs latency).


In [ ]:
# Cell — Aggregate
%cd /content/vtp-eval
!python -m vtp_eval.utils.result_io aggregate results/ --output results/summary.csv
import pandas as pd
df = pd.read_csv("results/summary.csv")
df


In [ ]:
# Cell — Plot
import matplotlib.pyplot as plt
import pandas as pd
df = pd.read_csv("results/summary.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for _, row in df.iterrows():
    axes[0].scatter(row["tflops_prefill"], row["pope_f1"], s=80)
    axes[0].annotate(row["method"], (row["tflops_prefill"], row["pope_f1"]),
                     fontsize=9, xytext=(5, 5), textcoords="offset points")
    axes[1].scatter(row["latency_ms_mean"], row["pope_f1"], s=80)
    axes[1].annotate(row["method"], (row["latency_ms_mean"], row["pope_f1"]),
                     fontsize=9, xytext=(5, 5), textcoords="offset points")
axes[0].set(xlabel="Prefill TFLOPs (theoretical)", ylabel="POPE F1",
            title="Accuracy vs Compute")
axes[1].set(xlabel="Latency per sample (ms)", ylabel="POPE F1",
            title="Accuracy vs Latency")
fig.tight_layout()
plt.savefig("results/scatter.png", dpi=150)
plt.show()
